# Test: Full Pipeline

Runs the complete pipeline (quality cleaner -> language editor) on a real file.
Shows side-by-side comparison and quality metrics.
Read-only — no files are written.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import re
from pathlib import Path
from dotenv import load_dotenv
load_dotenv('../.env')

from openai import OpenAI
from agent_loader import load_agents_from_dir
from pipeline_config import load_pipeline_config
from pipeline import run_pipeline

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
agents = load_agents_from_dir(Path('../agents'))
config = load_pipeline_config(Path('../pipeline.md'))
print(f'Pipeline loaded: {[a.name for a in agents]}')
print(f'Brand: {config.brand_name}, Locations: {config.locations}')

In [ ]:
sample_file = Path('../../apps/deck-repair/src/data/generated_content/how_to_choose_the_best_deck_repair_contractor_in_portland.md')
original = sample_file.read_text(encoding='utf-8')
print(f'Input: {sample_file.name} ({len(original)} chars)')

In [ ]:
result = run_pipeline(
    content=original,
    agents=agents,
    pipeline_config=config,
    client=client,
)
print(f'Output: {len(result)} chars ({len(result)-len(original):+d})')

In [ ]:
print('=== ORIGINAL (first 800 chars) ===')
print(original[:800])
print('\n=== RESULT (first 800 chars) ===')
print(result[:800])

In [ ]:
# Quality metrics
citation_pattern = r'\[Source:[^\]]+\]'
malformed_pattern = r'For (related services|comprehensive solutions),'

metrics = {
    'Citation artifacts': (
        len(re.findall(citation_pattern, original)),
        len(re.findall(citation_pattern, result)),
    ),
    'Malformed link injections': (
        len(re.findall(malformed_pattern, original)),
        len(re.findall(malformed_pattern, result)),
    ),
    'Character count': (len(original), len(result)),
}

print('Quality Metrics:')
print(f'{"Metric":<30} {"Before":>8} {"After":>8}')
print('-' * 50)
for name, (before, after) in metrics.items():
    status = 'OK' if after <= before else 'WARN'
    print(f'{name:<30} {before:>8} {after:>8}  [{status}]')